# mCREAM Graph Module Ensemble Analysis

M Concept-Concept blocks (one per expert graph), shared backbone/side-channel/task-head.
Aggregation at **concept level** c, not prediction level y.

Includes:
- Perturbed graph visualization
- Task & Concept accuracy (table + plot)
- CCI and intervention curves
- Single-edge perturbation results


In [ ]:
import pandas as pd, numpy as np, ast, torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

ACTION_COLOR  = {'deletion':'#e74c3c','addition':'#2ecc71','reversal':'#3498db'}
EXPERT_COLORS = ['#1f77b4','#d62728','#2ca02c','#ff7f0e','#9467bd']
LEVEL_ALPHA   = {'low':0.45,'medium':0.70,'high':1.0}

EXPERIMENTS_ROOT = Path('/home/dani00003/mCREAM/experiments')
GRAPHS_ROOT      = Path('/home/dani00003/mCREAM/data/FashionMNIST')
DAG_CFMNIST      = GRAPHS_ROOT / 'Complete_Concept_FMNIST_DAG.csv'

DATASETS = ['Complete_Concept_FMNIST']
ACTIONS  = ['deletion','addition','reversal']
LEVELS   = ['low','medium','high']
N_EXPERTS = 5
print(f'Root exists: {EXPERIMENTS_ROOT.exists()}')


## 1. Load Results


In [ ]:
def load_graph_ensemble(root):
    rows = []
    for ds in DATASETS:
        ens_dir = root / ds / 'train_cbm' / 'mCREAM_GraphEnsemble'
        if not ens_dir.exists(): print(f'  [SKIP] {ens_dir}'); continue
        for exp_dir in sorted(ens_dir.iterdir()):
            if not exp_dir.is_dir(): continue
            name = exp_dir.name  # graph_ensemble_{action}_{level}
            p = name.split('_')
            try:
                # graph_ensemble_{action}_{level}
                action = p[2]; level = p[3]
            except IndexError: continue
            for seed_dir in sorted(exp_dir.glob('seed_*/lightning_logs/version_*')):
                seed = int(seed_dir.parent.parent.name.split('_')[1])
                for csv_f in sorted(seed_dir.glob('*.csv')):
                    if any(x in csv_f.name for x in ['perc_','_set_','intervention','exogenous']): continue
                    try:
                        df = pd.read_csv(csv_f)
                        df['dataset']=ds; df['action']=action
                        df['noise_level']=level; df['seed']=seed
                        rows.append(df)
                    except Exception as e: print(f'ERR {csv_f}: {e}')
    if not rows: print('No results found'); return pd.DataFrame()
    df = pd.concat(rows, ignore_index=True)
    print(f'Loaded {len(df)} rows | actions={sorted(df.action.unique())} | levels={sorted(df.noise_level.unique())}')
    return df

def load_interventions(root, model_folder='mCREAM_GraphEnsemble'):
    rows = []
    for ds in DATASETS:
        ens_dir = root / ds / 'train_cbm' / model_folder
        if not ens_dir.exists(): continue
        for csv_f in ens_dir.rglob('intervention_results.csv'):
            try:
                df = pd.read_csv(csv_f)
                parts = csv_f.parts
                exp_name = seed = None
                for i,p in enumerate(parts):
                    if p == model_folder and i+1 < len(parts): exp_name = parts[i+1]
                    if p.startswith('seed_'): seed = int(p.split('_')[1])
                if exp_name is None: continue
                p = exp_name.split('_')
                try: action=p[2]; level=p[3]
                except: continue
                df['dataset']=ds; df['action']=action; df['noise_level']=level; df['seed']=seed
                rows.append(df)
            except Exception as e: print(f'ERR {csv_f}: {e}')
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def load_gt_cream(root):
    rows = []
    for ds,model,exp in [('Complete_Concept_FMNIST','Standard_FashionMNIST','CREAM_best_cfmnist')]:
        md = root/ds/'train_cbm'/model/exp/'last_metrics'
        if not md.exists(): continue
        for f in sorted(md.glob('*.csv')):
            df = pd.read_csv(f); df['dataset']=ds; rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

ens_df    = load_graph_ensemble(EXPERIMENTS_ROOT)
interv_df = load_interventions(EXPERIMENTS_ROOT)
gt_df     = load_gt_cream(EXPERIMENTS_ROOT)
if len(gt_df)>0:
    for ds in gt_df['dataset'].unique():
        r=gt_df[gt_df['dataset']==ds].iloc[0]
        print(f'GT CREAM {ds}: acc={r.get("test_task_accuracy",float("nan")):.4f}  concept_acc={r.get("test_concept_accuracy","N/A")}  CCI={r.get("CCI","N/A")}')


## 2. Perturbed Graph Visualization (5 seeds per noise level)


In [ ]:
def plot_perturbed_graphs(action, level, n_seeds=5):
    gt_df_dag = pd.read_csv(DAG_CFMNIST, index_col=0)
    K, T = 11, 10
    concepts = list(gt_df_dag.columns[:K])
    gt_vals  = (gt_df_dag.values != 0).astype(int)
    gt_u2c   = gt_vals[:K, :K]

    graphs_dir = GRAPHS_ROOT / 'expert_graphs' / 'ensemble' / f'{action}_{level}'
    if not graphs_dir.exists(): print(f'Graphs not found: {graphs_dir}'); return

    fig, axes = plt.subplots(1, n_seeds+1, figsize=(2.5*(n_seeds+1), 3))
    fig.suptitle(f'u2c graphs — {action} / {level}\n'
                 f'Blue=kept  Red=deleted  Green=added  Grey=absent',
                 fontsize=10, fontweight='bold')

    def make_rgb(p_block):
        rgb = np.zeros((*p_block.shape, 3))
        for r in range(p_block.shape[0]):
            for c in range(p_block.shape[1]):
                if   gt_u2c[r,c]==1 and p_block[r,c]==1: rgb[r,c]=[0.2,0.4,0.8]
                elif gt_u2c[r,c]==0 and p_block[r,c]==1: rgb[r,c]=[0.2,0.8,0.2]
                elif gt_u2c[r,c]==1 and p_block[r,c]==0: rgb[r,c]=[0.9,0.2,0.2]
                else: rgb[r,c]=[0.93,0.93,0.93]
        return rgb

    # GT reference
    gt_rgb = np.zeros((*gt_u2c.shape, 3))
    for r in range(K):
        for c in range(K):
            gt_rgb[r,c] = [0.2,0.4,0.8] if gt_u2c[r,c]==1 else [0.93,0.93,0.93]
    axes[0].imshow(gt_rgb, aspect='auto', interpolation='nearest')
    axes[0].set_title('GT', fontweight='bold', fontsize=8)
    for spine in axes[0].spines.values(): spine.set_edgecolor('gold'); spine.set_linewidth(3)
    axes[0].set_xticks([]); axes[0].set_yticks([])

    for s in range(n_seeds):
        f = graphs_dir / 'u2c' / f'expert_{s}.pt'
        if not f.exists(): axes[s+1].axis('off'); continue
        p = torch.load(f, weights_only=True).float().numpy()
        axes[s+1].imshow(make_rgb(p), aspect='auto', interpolation='nearest')
        axes[s+1].set_title(f'Seed {s}', fontsize=8)
        axes[s+1].set_xticks([]); axes[s+1].set_yticks([])

    plt.tight_layout()
    plt.savefig(f'graph_ensemble_perturbed_{action}_{level}.png', dpi=150, bbox_inches='tight')
    plt.show()

for action in ACTIONS:
    for level in LEVELS:
        plot_perturbed_graphs(action, level)


## 3. Summary Table — Task & Concept Accuracy, CCI


In [ ]:
if len(ens_df)==0: print('No data')
else:
    KEY=['test_task_accuracy','test_concept_accuracy','CCI','PFI_concept_importance',
         'intervention_acc_0','intervention_acc_max']
    av=[c for c in KEY if c in ens_df.columns]
    means=ens_df.groupby(['dataset','action','noise_level'])[av].mean()
    stds =ens_df.groupby(['dataset','action','noise_level'])[av].std()
    sm=pd.DataFrame(index=means.index)
    for c in av: sm[c]=means[c].map('{:.4f}'.format)+' ± '+stds[c].map('{:.4f}'.format)
    if len(gt_df)>0:
        print('--- GT CREAM baseline ---')
        for c in av:
            if c in gt_df.columns: print(f'  {c:35s} {gt_df[c].mean():.4f}')
    for ds in DATASETS:
        if ds not in means.index.get_level_values('dataset'): continue
        print(f'\n--- {ds} ---')
        display(sm.xs(ds,level='dataset'))


## 4. Task & Concept Accuracy — Line Plot


In [ ]:
if len(ens_df)==0: print('No data')
else:
    lnum={'low':0.25,'medium':0.50,'high':0.75}
    for ds in DATASETS:
        d=ens_df[ens_df['dataset']==ds].copy()
        if d.empty: continue
        d['noise_prob']=d['noise_level'].map(lnum)
        fig,axes=plt.subplots(1,2,figsize=(12,4.5))
        fig.suptitle(f'{ds}\nmCREAM Graph Ensemble: Accuracy vs Noise Level',fontsize=12,fontweight='bold')
        for ax,metric,ylabel in [
            (axes[0],'test_task_accuracy','Task Accuracy'),
            (axes[1],'test_concept_accuracy','Concept Accuracy')]:
            if metric not in d.columns: continue
            agg=d.groupby(['action','noise_prob'])[metric].agg(['mean','std']).reset_index()
            for action in ACTIONS:
                sub=agg[agg['action']==action]
                if sub.empty: continue
                c=ACTION_COLOR[action]
                ax.plot(sub['noise_prob'],sub['mean'],color=c,marker='o',markersize=8,lw=2,label=action)
                ax.fill_between(sub['noise_prob'],sub['mean']-sub['std'],sub['mean']+sub['std'],alpha=0.12,color=c)
            if len(gt_df)>0 and metric in gt_df.columns:
                sub2=gt_df[gt_df['dataset']==ds]
                if not sub2.empty:
                    v=sub2[metric].mean()
                    ax.axhline(v,color='black',lw=2,ls='--',label=f'CREAM GT: {v:.4f}')
            ax.set_xlabel('Noise probability',fontsize=9); ax.set_ylabel(ylabel,fontsize=9)
            ax.set_xticks([0.25,0.50,0.75]); ax.set_xticklabels(['low','medium','high'])
            ax.legend(title='Noise type',fontsize=8,framealpha=0.9); ax.tick_params(labelsize=8)
        plt.tight_layout()
        plt.savefig(f'graph_ensemble_accuracy_{ds}.png',dpi=150,bbox_inches='tight')
        plt.show()


## 5. CCI Plot


In [ ]:
if len(ens_df)==0 or 'CCI' not in ens_df.columns: print('No CCI data')
else:
    lnum={'low':0.25,'medium':0.50,'high':0.75}
    for ds in DATASETS:
        d=ens_df[ens_df['dataset']==ds].dropna(subset=['CCI']).copy()
        if d.empty: continue
        d['noise_prob']=d['noise_level'].map(lnum)
        agg=d.groupby(['action','noise_prob'])['CCI'].agg(['mean','std']).reset_index()
        fig,ax=plt.subplots(figsize=(7,4.5))
        fig.suptitle(f'{ds}\nCCI (Concept Channel Importance) vs Noise Level',fontsize=12,fontweight='bold')
        for action in ACTIONS:
            sub=agg[agg['action']==action]
            if sub.empty: continue
            c=ACTION_COLOR[action]
            ax.plot(sub['noise_prob'],sub['mean'],color=c,marker='o',markersize=8,lw=2,label=action)
            ax.fill_between(sub['noise_prob'],sub['mean']-sub['std'],sub['mean']+sub['std'],alpha=0.12,color=c)
        if len(gt_df)>0 and 'CCI' in gt_df.columns:
            sub2=gt_df[gt_df['dataset']==ds]
            if not sub2.empty and pd.notna(sub2['CCI'].mean()):
                v=sub2['CCI'].mean()
                ax.axhline(v,color='black',lw=2,ls='--',label=f'CREAM GT: {v:.3f}')
        ax.axhline(0.5,color='red',lw=1.2,ls=':',alpha=0.6,label='threshold=0.5')
        ax.set_xlabel('Noise probability',fontsize=9); ax.set_ylabel('CCI',fontsize=9)
        ax.set_xticks([0.25,0.50,0.75]); ax.set_xticklabels(['low','medium','high'])
        ax.legend(title='Noise type',fontsize=9,framealpha=0.9); ax.tick_params(labelsize=8)
        plt.tight_layout()
        plt.savefig(f'graph_ensemble_cci_{ds}.png',dpi=150,bbox_inches='tight')
        plt.show()


## 6. Intervention Curves


In [ ]:
if len(interv_df)==0: print('No intervention data yet.')
else:
    for ds in DATASETS:
        for action in ACTIONS:
            sub=interv_df[(interv_df['dataset']==ds)&(interv_df['action']==action)]
            if sub.empty: continue
            fig,axes=plt.subplots(1,len(LEVELS),figsize=(5*len(LEVELS),4.5),sharey=True)
            fig.suptitle(f'{ds} — {action} noise\nmCREAM Graph Ensemble intervention curves',fontsize=11,fontweight='bold')
            for ax,level in zip(axes,LEVELS):
                lv=sub[sub['noise_level']==level]
                if lv.empty: ax.set_title(level); continue
                agg=lv.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
                ax.plot(agg['num_interventions'],agg['mean'],color=ACTION_COLOR[action],lw=2.5,marker='o',markersize=5)
                ax.fill_between(agg['num_interventions'],agg['mean']-agg['std'],agg['mean']+agg['std'],alpha=0.15,color=ACTION_COLOR[action])
                # GT CREAM intervention reference
                if len(gt_df)>0 and 'intervention_results' in gt_df.columns:
                    raw=gt_df[gt_df['dataset']==ds]['intervention_results'].iloc[0]
                    if isinstance(raw,str):
                        gt_interv=pd.DataFrame([{'n':e['num_interventions'],'acc':e['metrics']['test_task_accuracy']}
                                                 for e in ast.literal_eval(raw) if not e.get('group_interventions',False)])
                        if not gt_interv.empty:
                            ax.plot(gt_interv['n'],gt_interv['acc'],color='black',lw=2,ls='--',marker='s',markersize=4,label='CREAM GT')
                ax.axhline(1.0,color='gray',ls=':',alpha=0.4,lw=1)
                ax.set_title(f'{level} noise',fontsize=10); ax.set_xlabel('Number of interventions',fontsize=9)
                ax.set_ylabel('Task Accuracy' if level==LEVELS[0] else '',fontsize=9)
                ax.legend(fontsize=7,loc='lower right'); ax.tick_params(labelsize=8)
            plt.tight_layout()
            plt.savefig(f'graph_ensemble_interv_{ds}_{action}.png',dpi=150,bbox_inches='tight')
            plt.show()


## 7. Single-Edge Perturbation Results

For the graph ensemble: each expert gets a graph with ONE edge changed.
CCI vs accuracy scatter — which edge matters most?


In [ ]:
def load_single_edge_graph_ensemble(root):
    rows=[]
    for ds in DATASETS:
        ens_dir=root/ds/'train_cbm'/'mCREAM_GraphEnsemble'
        if not ens_dir.exists(): continue
        for exp_dir in sorted(ens_dir.iterdir()):
            if not exp_dir.is_dir(): continue
            name=exp_dir.name
            if not (name.startswith('del_edge_') or name.startswith('add_edge_')): continue
            ptype='del' if name.startswith('del') else 'add'
            edge_label=name[4:]  # remove 'del_' or 'add_'
            for seed_dir in sorted(exp_dir.glob('seed_*/lightning_logs/version_*')):
                for csv_f in sorted(seed_dir.glob('*.csv')):
                    if any(x in csv_f.name for x in ['perc_','_set_','intervention','exogenous']): continue
                    try:
                        df=pd.read_csv(csv_f)
                        df['dataset']=ds; df['perturb_type']=ptype; df['edge_label']=edge_label
                        rows.append(df)
                    except: pass
    if not rows: print('No single-edge results for graph ensemble yet'); return pd.DataFrame()
    df=pd.concat(rows,ignore_index=True)
    print(f'Loaded {len(df)} single-edge graph ensemble rows')
    return df

single_df=load_single_edge_graph_ensemble(EXPERIMENTS_ROOT)


In [ ]:
if len(single_df)==0: print('No single-edge graph ensemble data yet.')
else:
    for ds in DATASETS:
        d=single_df[single_df['dataset']==ds]
        if d.empty or 'CCI' not in d.columns: continue
        fig,ax=plt.subplots(figsize=(9,6))
        fig.suptitle(f'{ds}\nmCREAM Graph Ensemble: Single-Edge Perturbation\n'
                     f'Red=deletion  Green=addition  Star=GT CREAM',fontsize=11,fontweight='bold')
        del_d=d[d['perturb_type']=='del'].dropna(subset=['CCI','test_task_accuracy'])
        add_d=d[d['perturb_type']=='add'].dropna(subset=['CCI','test_task_accuracy'])
        if not del_d.empty:
            ax.scatter(del_d['CCI'],del_d['test_task_accuracy'],
                       color='#e74c3c',s=80,alpha=0.8,zorder=5,label=f'Deletion ({len(del_d)})')
            for _,row in del_d.nsmallest(3,'test_task_accuracy').iterrows():
                ax.annotate(row['edge_label'].replace('edge_','').replace('_','→',1),
                            (row['CCI'],row['test_task_accuracy']),fontsize=7,xytext=(5,3),textcoords='offset points')
        if not add_d.empty:
            ax.scatter(add_d['CCI'],add_d['test_task_accuracy'],
                       color='#2ecc71',s=80,alpha=0.8,marker='^',zorder=5,label=f'Addition ({len(add_d)})')
        if len(gt_df)>0 and 'CCI' in gt_df.columns and 'test_task_accuracy' in gt_df.columns:
            sub2=gt_df[gt_df['dataset']==ds]
            if not sub2.empty:
                ax.scatter([sub2['CCI'].mean()],[sub2['test_task_accuracy'].mean()],
                           color='black',s=250,marker='*',zorder=10,label=f'GT CREAM')
        ax.axhline(0.5,color='red',ls=':',alpha=0.3); ax.axvline(0.5,color='red',ls=':',alpha=0.3)
        ax.set_xlabel('CCI',fontsize=10); ax.set_ylabel('Task Accuracy',fontsize=10)
        ax.legend(fontsize=9,framealpha=0.9); ax.tick_params(labelsize=8)
        plt.tight_layout()
        plt.savefig(f'graph_ensemble_single_edge_{ds}.png',dpi=150,bbox_inches='tight')
        plt.show()


## 8. Export Summary


In [ ]:
if len(ens_df)>0:
    key=['test_task_accuracy','test_concept_accuracy','CCI','PFI_concept_importance','intervention_acc_max']
    av=[c for c in key if c in ens_df.columns]
    agg=ens_df.groupby(['dataset','action','noise_level'])[av].agg(['mean','std']).round(4)
    agg.to_csv('graph_ensemble_summary.csv'); print('Saved: graph_ensemble_summary.csv'); display(agg)
